 Data Cleaning — MyAnimeList Anime Dataset (2023)

In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv('anime-dataset-2023.csv', encoding='utf-8')
df.describe() #overview of dataset


,anime_id,Popularity,Favorites,Members
count,24905.000000,24905.000000,24905.000000,2.490500e+04
mean,29776.709014,12265.388356,432.595222,3.710496e+04
std,17976.076290,7187.428393,4353.181647,1.568252e+05
min,1.000000,0.000000,0.000000,0.000000e+00
25%,10507.000000,6040.000000,0.000000,2.090000e+02
50%,34628.000000,12265.000000,1.000000,1.056000e+03
75%,45240.000000,18491.000000,18.000000,9.326000e+03
max,55735.000000,24723.000000,217606.000000,3.744541e+06


In [ ]:
df.info() #few numeric column are in string datatype

<class 'pandas.DataFrame'>
RangeIndex: 24905 entries, 0 to 24904
Data columns (total 24 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   anime_id      24905 non-null  int64
 1   Name          24905 non-null  str  
 2   English name  24905 non-null  str  
 3   Other name    24905 non-null  str  
 4   Score         24905 non-null  str  
 5   Genres        24905 non-null  str  
 6   Synopsis      24905 non-null  str  
 7   Type          24905 non-null  str  
 8   Episodes      24905 non-null  str  
 9   Aired         24905 non-null  str  
 10  Premiered     24905 non-null  str  
 11  Status        24905 non-null  str  
 12  Producers     24905 non-null  str  
 13  Licensors     24905 non-null  str  
 14  Studios       24905 non-null  str  
 15  Source        24905 non-null  str  
 16  Duration      24905 non-null  str  
 17  Rating        24905 non-null  str  
 18  Rank          24905 non-null  str  
 19  Popularity    24905 non-null  int64


In [3]:
df['Score'].count() #no nulls but datatype should be int or float

np.int64(24905)

In [4]:
pd.to_numeric(df['Score'], errors='coerce')

0        8.75
1        8.38
2        8.22
3        7.25
4        6.94
         ... 
24900     NaN
24901     NaN
24902     NaN
24903     NaN
24904     NaN
Name: Score, Length: 24905, dtype: float64

In [5]:
score_numeric = pd.to_numeric(df['Score'], errors='coerce')

In [6]:
score_numeric.isnull().sum()

np.int64(9213)

In [7]:
df['Score'] = score_numeric

In [8]:
df['Score'].isnull().sum() #double check of nulls or any dirty data

np.int64(9213)

In [9]:
(df['Rank']== 'UNKNOWN').sum()

np.int64(4612)

In [10]:
(df['Members']== 'UNKNOWN').sum()

np.int64(0)

In [11]:
rank_numeric = pd.to_numeric(df['Rank'], errors='coerce')

In [12]:
df['Rank'] = rank_numeric

In [13]:
episode_numeric = pd.to_numeric(df['Episodes'],errors='coerce')

In [14]:

df['Episodes']=episode_numeric

In [15]:
df['Episodes'].isnull().sum()

np.int64(611)

cleaning 'Type', 'Source', 'Rating', 'Producers', 'Licensors', 'Studios', 'Premiered', 'Other name' with loop altogether after checking the data one by one



In [16]:
def clean_unknown(df, column_name):
    return df[column_name].replace('UNKNOWN', np.nan)

In [17]:
columns_to_clean = ['Type', 'Source', 'Rating', 'Producers', 'Licensors', 'Studios', 'Premiered', 'Other name']

In [18]:
for column_name in columns_to_clean:
 df[column_name] = clean_unknown(df, column_name)

In [19]:
df['Duration'].str.contains('unknown', case=False, na=False).sum()

np.int64(663)

#the duration column had mix units with not needed words like 'per episode', made another clean version of Duration as duration_minute

In [20]:
def duration_to_minute(text):
    if pd.isna(text):
        return pd.NA
    matches = re.findall(r'(\d+)\s*(hr|min|sec)', text)
    total = 0
    for number, unit in matches:
        number = int(number)
        if unit == 'hr':
            total += number * 60
        elif unit == 'min':
            total += number
        elif unit == 'sec':
            total += number / 60
    return total

In [21]:
import re
df['Duration_minute'] = df['Duration'].apply(duration_to_minute)

#phrasing duration into mintues


In [22]:
df['Duration'].str.contains('UNKNOWN', case=False,na= False)

0        False
1        False
2        False
3        False
4        False
         ...  
24900     True
24901     True
24902     True
24903    False
24904    False
Name: Duration, Length: 24905, dtype: bool

In [23]:
df.loc[df['English name'].str.contains('unknown', case=False, na=False),['English name']]

,English name
5,UNKNOWN
7,UNKNOWN
8,UNKNOWN
13,UNKNOWN
19,UNKNOWN
...,...
24898,UNKNOWN
24899,UNKNOWN
24900,UNKNOWN
24903,UNKNOWN


In [24]:

english_name_clean = df['English name'].replace('UNKNOWN', np.nan)
english_name_clean = english_name_clean.fillna(df['Name'])

In [25]:
english_name_clean.isnull().sum()

np.int64(0)

In [26]:
df['English name']= english_name_clean

In [27]:
df['anime_id'].duplicated().sum() #checking for duplicate value in PK 

np.int64(0)

In [28]:
df.loc[df['Name'].str.contains('UNKNOWN',case=False)]

,anime_id,Name,English name,Other name,Score,Genres,Synopsis,Type,Episodes,Aired,...,Source,Duration,Rating,Rank,Popularity,Favorites,Scored By,Members,Image URL,Duration_minute
14933,38648,King of Prism: Shiny Seven Stars IV - Louis x ...,King of Prism: Shiny Seven Stars IV - Louis x ...,KING OF PRISM -Shiny Seven Stars- IV ルヰxシンxUnk...,NaN,Sports,The fourth movie in the four-part King of Pris...,Movie,1.0,"May 4, 2019",...,Original,Unknown,PG-13 - Teens 13 or older,17311.0,13306,0,UNKNOWN,815,https://cdn.myanimelist.net/img/sp/icon/apple-...,0.0


In [29]:
df['Genres'].str.contains('Unknown', case=False, na=False).sum()

np.int64(4929)

In [30]:
df['Genres'].unique()

<StringArray>
[                       'Action, Award Winning, Sci-Fi',
                                       'Action, Sci-Fi',
                            'Action, Adventure, Sci-Fi',
                 'Action, Drama, Mystery, Supernatural',
                     'Adventure, Fantasy, Supernatural',
                                               'Sports',
                               'Comedy, Drama, Romance',
                        'Comedy, Slice of Life, Sports',
                                        'Action, Drama',
                             'Drama, Mystery, Suspense',
 ...
                                           'Girls Love',
 'Adventure, Drama, Fantasy, Mystery, Sci-Fi, Suspense',
     'Action, Drama, Horror, Mystery, Sci-Fi, Suspense',
                          'Action, Avant Garde, Sci-Fi',
                      'Action, Fantasy, Sci-Fi, Sports',
                                      'Gourmet, Sci-Fi',
                         'Avant Garde, Drama, Suspense',
            

In [31]:
genre_dummies = df['Genres'].str.get_dummies(sep=', ')

In [32]:
df = pd.concat([df, genre_dummies], axis=1)

In [33]:
df.loc[df['Scored By'].str.contains('unknown', case=False)]

,anime_id,Name,English name,Other name,Score,Genres,Synopsis,Type,Episodes,Aired,...,Hentai,Horror,Mystery,Romance,Sci-Fi,Slice of Life,Sports,Supernatural,Suspense,UNKNOWN
1578,1739,Shibawanko no Wa no Kokoro,Shibawanko no Wa no Kokoro,しばわんこの和のこころ,NaN,UNKNOWN,Based on a japanese children`s book by Yoshie ...,TV,80.0,"Apr 5, 2006 to Mar 14, 2007",...,0,0,0,0,0,0,0,0,0,1
1699,1863,Silk Road Shounen Yuuto,Silk Road Shounen Yuuto,シルクロード少年 ユート,NaN,"Adventure, Fantasy","When a boy Yuto visits Qinghai in China, he is...",TV,26.0,"Sep 16, 2006 to Mar 24, 2007",...,0,0,0,0,0,0,0,0,0,0
2476,2701,Susie-chan to Marvy,Little Susie and Marvy,スージーちゃんとマービー,NaN,Comedy,No description available for this anime.,TV,104.0,"Apr 5, 1999 to Feb 3, 2000",...,0,0,0,0,0,0,0,0,0,0
2483,2708,Wankorobee,Wankorobee,わんころべえ,NaN,"Comedy, Fantasy",No description available for this anime.,TV,26.0,"Oct 6, 1996 to Mar 30, 1997",...,0,0,0,0,0,0,0,0,0,0
2508,2735,Mugen Senki Portriss,Mugen Senki Portriss,無限戦記ポトリス,NaN,"Action, Sci-Fi","In a raving world, legendary knights stood up....",TV,52.0,"Apr 5, 2003 to Mar 27, 2004",...,0,0,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24900,55731,Wu Nao Monu,Wu Nao Monu,无脑魔女,NaN,"Comedy, Fantasy, Slice of Life",No description available for this anime.,ONA,15.0,"Jul 4, 2023 to ?",...,0,0,0,0,0,1,0,0,0,0
24901,55732,Bu Xing Si: Yuan Qi,Blader Soul,捕星司·源起,NaN,"Action, Adventure, Fantasy",No description available for this anime.,ONA,18.0,"Jul 27, 2023 to ?",...,0,0,0,0,0,0,0,0,0,0
24902,55733,Di Yi Xulie,The First Order,第一序列,NaN,"Action, Adventure, Fantasy, Sci-Fi",No description available for this anime.,ONA,16.0,"Jul 19, 2023 to ?",...,0,0,0,0,1,0,0,0,0,0
24903,55734,Bokura no Saishuu Sensou,Bokura no Saishuu Sensou,僕らの最終戦争,NaN,UNKNOWN,A music video for the song Bokura no Saishuu S...,Music,1.0,"Apr 23, 2022",...,0,0,0,0,0,0,0,0,0,1


In [34]:
df['Scored By'].head(20) 

0      914193.0
1      206248.0
2      356739.0
3       42829.0
4        6413.0
5       86524.0
6       81747.0
7       12960.0
8       97878.0
9      368569.0
10    1883772.0
11    1226493.0
12      81992.0
13       1378.0
14     146597.0
15      53709.0
16      54857.0
17      79130.0
18      40492.0
19       7273.0
Name: Scored By, dtype: str

In [35]:
scoredby= df['Scored By'].replace('UNKNOWN',np.nan)

In [36]:
(scoredby).isnull().sum()

np.int64(9213)

In [37]:
df['Scored By']= scoredby

In [38]:
df['Scored By'] = df['Scored By'].astype(float)

In [39]:
df['Image URL'].str.contains('unknown', case=False).sum() #no cleaning needed

np.int64(0)

In [40]:
df['Status'].str.contains('unknown', case=False).sum()

np.int64(0)

In [41]:
df['Status'].unique() #no cleaning needed

<StringArray>
['Finished Airing', 'Currently Airing', 'Not yet aired']
Length: 3, dtype: str

In [ ]:
df['Synopsis'].str.contains('UNKNOWN',case=False).sum()#no replacement needed as there is no UNKNOWN column alone.

np.int64(247)

In [43]:
df['Aired'].unique()

<StringArray>
[ 'Apr 3, 1998 to Apr 24, 1999',                  'Sep 1, 2001',
  'Apr 1, 1998 to Sep 30, 1998',  'Jul 3, 2002 to Dec 25, 2002',
 'Sep 30, 2004 to Sep 29, 2005',  'Apr 6, 2005 to Mar 19, 2008',
 'Apr 15, 2005 to Sep 27, 2005', 'Sep 11, 2002 to Sep 10, 2003',
 'Apr 17, 2004 to Feb 18, 2006',  'Apr 7, 2004 to Sep 28, 2005',
 ...
                 'Nov 19, 2019',                 'Jan 22, 2016',
            'Mar 10, 2023 to ?',            'May 31, 2023 to ?',
            'May 16, 2023 to ?',                  'Jul 3, 2014',
                 'Feb 11, 2015',            'Jul 27, 2023 to ?',
            'Jul 19, 2023 to ?',                 'Apr 23, 2022']
Length: 15213, dtype: str

In [44]:
aired_split = df['Aired'].str.split(' to ', expand=True)
df['Aired_start'] = aired_split[0]
df['Aired_end'] = aired_split[1]

In [45]:
df['Aired_start'] = pd.to_datetime(df['Aired_start'], errors='coerce')

In [46]:
df['Aired_end'] = df['Aired_end'].replace('?', np.nan)
df['Aired_end'] = pd.to_datetime(df['Aired_end'], errors='coerce')

In [47]:
df[['Aired', 'Aired_start', 'Aired_end', 'Status']].head(10)

,Aired,Aired_start,Aired_end,Status
0,"Apr 3, 1998 to Apr 24, 1999",1998-04-03,1999-04-24,Finished Airing
1,"Sep 1, 2001",2001-09-01,NaT,Finished Airing
2,"Apr 1, 1998 to Sep 30, 1998",1998-04-01,1998-09-30,Finished Airing
3,"Jul 3, 2002 to Dec 25, 2002",2002-07-03,2002-12-25,Finished Airing
4,"Sep 30, 2004 to Sep 29, 2005",2004-09-30,2005-09-29,Finished Airing
5,"Apr 6, 2005 to Mar 19, 2008",2005-04-06,2008-03-19,Finished Airing
6,"Apr 15, 2005 to Sep 27, 2005",2005-04-15,2005-09-27,Finished Airing
7,"Sep 11, 2002 to Sep 10, 2003",2002-09-11,2003-09-10,Finished Airing
8,"Apr 17, 2004 to Feb 18, 2006",2004-04-17,2006-02-18,Finished Airing
9,"Apr 7, 2004 to Sep 28, 2005",2004-04-07,2005-09-28,Finished Airing


In [ ]:
df = df.drop(columns=['UNKNOWN']) #dropped this column which was created from genre column

In [49]:
df[df['Premiered'].isnull()]['Type'].value_counts()

Type
Movie      4381
OVA        4076
ONA        3533
Music      2686
Special    2558
TV         2091
Name: count, dtype: int64

In [52]:
df.shape

(24905, 48)

In [54]:
df.isnull().sum()

anime_id               0
Name                   0
English name           0
Other name           128
Score               9213
Genres                 0
Synopsis               0
Type                  74
Episodes             611
Aired                  0
Premiered          19399
Status                 0
Producers          13350
Licensors          20170
Studios            10526
Source                 0
Duration               0
Rating               669
Rank                4612
Popularity             0
Favorites              0
Scored By           9213
Members                0
Image URL              0
Duration_minute        0
Action                 0
Adventure              0
Avant Garde            0
Award Winning          0
Boys Love              0
Comedy                 0
Drama                  0
Ecchi                  0
Erotica                0
Fantasy                0
Girls Love             0
Gourmet                0
Hentai                 0
Horror                 0
Mystery                0


summary -
cleaned data by replacing 'unknown' str with Nan. 
changed few numeric col dtype into float/int
fixed Aired dataype to datetime
new col added out of genre in the boolean form 
duration unit was fixed into one unit i.e minutes


In [55]:
df.to_csv('cleaned_data.csv', index=False, encoding='utf-8-sig')